# Energy Market Price Forecasting – Statistical Regression Model

This notebook demonstrates a **Linear Regression** approach to forecasting day-ahead electricity prices.

The dataset is a realistic synthetic hourly profile for a single day (24 observations) with the following features:

| Feature | Description |
|---|---|
| `hour_of_day` | Hour (0–23) |
| `demand_mwh` | System demand in MWh |
| `gas_price_eur` | Day-ahead gas price in €/MWh |
| `wind_mwh` | Wind generation forecast in MWh |
| `price_eur_mwh` | **Target** – day-ahead electricity price in €/MWh |

> **Why train/test split matters here:** Energy price models must generalise to future, unseen hours. Evaluating on training data gives an overly optimistic R², hiding whether the model will hold up in live trading.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

np.random.seed(42)

## 1. Build a Realistic Synthetic Dataset

Prices follow a realistic day-ahead electricity profile:
- Low prices overnight (off-peak demand, excess wind)
- Two peaks at morning (~08:00) and evening (~18:00) ramp-up
- Gas price and demand amplify the price; wind generation suppresses it

In [ ]:
hours = np.arange(24)

# Demand profile (MWh) – morning and evening peaks
demand = 30_000 + 8_000 * np.sin(np.pi * (hours - 6) / 12) ** 2

# Wind generation (MWh) – higher at night
wind = 6_000 + 2_000 * np.cos(np.pi * hours / 12)

# Gas price (€/MWh) – slight intraday variation around €85
gas_price = 85 + 3 * np.sin(2 * np.pi * hours / 24)

# Electricity price: driven by demand and gas, suppressed by wind
price = (
    20
    + 0.002 * demand
    - 0.003 * wind
    + 0.4 * gas_price
    + np.random.normal(0, 2, 24)   # market noise
)

df = pd.DataFrame({
    "hour_of_day": hours,
    "demand_mwh": demand,
    "wind_mwh": wind,
    "gas_price_eur": gas_price,
    "price_eur_mwh": price,
})

print(df.to_string(index=False, float_format="{:.2f}".format))

## 2. Train / Test Split

We hold out 20 % of hours as the **test set** so that all evaluation metrics reflect performance on unseen data.

In [ ]:
features = ["hour_of_day", "demand_mwh", "wind_mwh", "gas_price_eur"]
target = "price_eur_mwh"

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

## 3. Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept (b0):", round(model.intercept_, 4))
for name, coef in zip(features, model.coef_):
    print(f"  {name:20s}: {coef:.6f}")

## 4. Evaluate on the Test Set

Three complementary metrics:

| Metric | Interpretation |
|---|---|
| **R²** | Proportion of variance explained (1.0 = perfect) |
| **MAE** | Average absolute error in €/MWh |
| **RMSE** | Error metric that penalises large spikes (critical for trading P&L) |

In [ ]:
y_pred_test = model.predict(X_test)

r2   = r2_score(y_test, y_pred_test)
mae  = mean_absolute_error(y_test, y_pred_test)
rmse = root_mean_squared_error(y_test, y_pred_test)

print(f"Test R²   : {r2:.4f}")
print(f"Test MAE  : {mae:.2f} €/MWh")
print(f"Test RMSE : {rmse:.2f} €/MWh")

## 5. Visualise Results

Two panels:
1. **Left** – Actual vs. Predicted scatter; a perfect model would place all points on the diagonal.
2. **Right** – Full 24-hour price profile showing where the model fits well and where it misses.

In [ ]:
y_pred_all = model.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel 1: Actual vs Predicted scatter ---
ax = axes[0]
ax.scatter(y_test, y_pred_test, color="steelblue", edgecolors="white", s=80, zorder=3)
lims = [min(y.min(), y_pred_all.min()) - 1, max(y.max(), y_pred_all.max()) + 1]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect fit")
ax.set_xlabel("Actual Price (€/MWh)")
ax.set_ylabel("Predicted Price (€/MWh)")
ax.set_title("Actual vs. Predicted (Test Set)")
ax.legend()
ax.set_xlim(lims); ax.set_ylim(lims)

# --- Panel 2: Full day price profile ---
ax2 = axes[1]
ax2.plot(hours, y, "o-", color="steelblue", label="Actual", linewidth=2)
ax2.plot(hours, y_pred_all, "s--", color="tomato", label="Predicted", linewidth=2)
ax2.set_xlabel("Hour of Day")
ax2.set_ylabel("Price (€/MWh)")
ax2.set_title("Day-Ahead Electricity Price – 24h Profile")
ax2.set_xticks(hours)
ax2.legend()

plt.tight_layout()
plt.savefig("price_forecast_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to price_forecast_results.png")

## 6. Predict for a New Hour

Forecast the day-ahead price for a hypothetical peak hour: **hour 18**, demand **38 000 MWh**, wind **4 500 MWh**, gas **€88/MWh**.

In [ ]:
new_obs = np.array([[18, 38_000, 4_500, 88]])
predicted_price = model.predict(new_obs)[0]
print(f"Forecast for hour 18 (peak demand, low wind): {predicted_price:.2f} €/MWh")